# Lab – Step 3: Predicate Alignment Using SPARQL

**Goal:** For each predicate in our private KB, find the best matching Wikidata property via SPARQL and declare the appropriate alignment:

| Relationship | RDF Axiom |
|---|---|
| Identical meaning | `prop:X owl:equivalentProperty wdt:PYY` |
| Private is narrower | `prop:X rdfs:subPropertyOf wdt:PYY` |
| Private is broader | `wdt:PYY rdfs:subPropertyOf prop:X` |
| Related (not exact) | `prop:X skos:relatedMatch wdt:PYY` |

**Two SPARQL strategies used:**
1. **Label search** — filter Wikidata properties whose label contains a keyword
2. **Instance-based validation** — given two aligned entities `(wd:s, wd:o)`, list all actual Wikidata properties connecting them

> **Note:** Wikidata's public SPARQL endpoint has a rate limit. The notebook adds delays and caches results to be polite.

## 0. Install & import

In [1]:
!pip install rdflib requests pandas SPARQLWrapper --quiet

In [2]:
import time
import requests
import pandas as pd
from SPARQLWrapper import SPARQLWrapper, JSON
from rdflib import Graph, Namespace, URIRef, Literal, RDF, RDFS, OWL, XSD

MUS   = Namespace("http://example.org/music/")
PROP  = Namespace("http://example.org/music/prop/")
WD    = Namespace("http://www.wikidata.org/entity/")
WDT   = Namespace("http://www.wikidata.org/prop/direct/")
SKOS  = Namespace("http://www.w3.org/2004/02/skos/core#")

# Load enriched KB from Step 2
g = Graph()
g.parse("music_kb_linked.ttl", format="turtle")

for prefix, ns in [("mus",MUS),("prop",PROP),("wd",WD),("wdt",WDT),
                   ("owl",OWL),("rdfs",RDFS),("skos",SKOS),("xsd",XSD)]:
    g.bind(prefix, ns)

# Wikidata SPARQL endpoint
sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
sparql.setReturnFormat(JSON)
sparql.addCustomHttpHeader("User-Agent", "MusicKB-Lab/3.0 (predicate-alignment)")

print(f"KB loaded: {len(g)} triples")

KB loaded: 848 triples


## 1. Collect all private predicates

In [3]:
# Collect predicates that live in our PROP namespace
private_predicates = set()
for s, p, o in g:
    if str(p).startswith(str(PROP)):
        private_predicates.add(p)

print(f"Private predicates found: {len(private_predicates)}")
for p in sorted(private_predicates):
    local = str(p).split("/")[-1]
    label = g.value(p, RDFS.label)
    print(f"  prop:{local:<25} -> {label}")

Private predicates found: 14
  prop:activeFrom                -> Year career started
  prop:albumRating               -> Critical rating 0-10
  prop:birthYear                 -> Year of birth
  prop:collaboratedWith          -> Artist collaborated with another artist
  prop:foundedYear               -> Year label founded
  prop:hasGenre                  -> Artist or album has genre
  prop:influencedBy              -> Artist was influenced by another artist
  prop:originatesFrom            -> Artist originates from country
  prop:partOfGenre               -> Genre is a subgenre of another genre
  prop:producedBy                -> Album produced by artist
  prop:releaseYear               -> Year album released
  prop:releasedAlbum             -> Artist released album
  prop:signedTo                  -> Artist signed to record label
  prop:wonAward                  -> Artist won award


## 2. SPARQL helpers

In [4]:
def run_sparql(query: str, retries: int = 3) -> list:
    """Execute a SPARQL query against Wikidata and return rows as list of dicts."""
    for attempt in range(retries):
        try:
            sparql.setQuery(query)
            results = sparql.query().convert()
            vars_   = results["head"]["vars"]
            rows    = []
            for binding in results["results"]["bindings"]:
                row = {v: binding[v]["value"] if v in binding else "" for v in vars_}
                rows.append(row)
            return rows
        except Exception as e:
            print(f"  [WARN] SPARQL attempt {attempt+1} failed: {e}")
            time.sleep(2 ** attempt)
    return []


def label_search_query(keyword: str, limit: int = 10) -> str:
    """Strategy 1: search Wikidata properties by label keyword."""
    kw = keyword.lower().replace('"', '')
    return f"""
SELECT DISTINCT ?prop ?propLabel WHERE {{
  ?prop a wikibase:Property .
  ?prop rdfs:label ?propLabel .
  FILTER(CONTAINS(LCASE(?propLabel), "{kw}"))
  FILTER(LANG(?propLabel) = "en")
}}
ORDER BY ?propLabel
LIMIT {limit}
"""


def instance_validation_query(wd_subject: str, wd_object: str, limit: int = 15) -> str:
    """Strategy 2: list real Wikidata properties between two aligned entities."""
    return f"""
SELECT DISTINCT ?prop ?propLabel WHERE {{
  wd:{wd_subject} ?prop wd:{wd_object} .
  ?propEntity wikibase:directClaim ?prop .
  ?propEntity rdfs:label ?propLabel .
  FILTER(LANG(?propLabel) = "en")
}}
LIMIT {limit}
"""

print("SPARQL helpers ready.")

SPARQL helpers ready.


## 3. Alignment plan

For each predicate we specify the keyword to search, the validated Wikidata PID, and the alignment type.  
The live SPARQL calls in Section 4 confirm and display the candidates.

In [5]:
# Format:
#   prop_local_name -> {
#       "keyword":      search term used,
#       "wd_pid":       best Wikidata property ID,
#       "wd_label":     Wikidata label,
#       "alignment":    "equivalent" | "subProperty" | "superProperty" | "related",
#       "rationale":    short explanation,
#       "instance_pair":(wd_subject_qid, wd_object_qid)  -- for validation query
#   }
ALIGNMENT_PLAN = {
    "wonAward": {
        "keyword": "award",
        "wd_pid": "P166",
        "wd_label": "award received",
        "alignment": "equivalent",
        "rationale": "P166 'award received' is the standard Wikidata property for winning an award. Semantically identical.",
        "instance_pair": ("Q11975", "Q41254"),
    },
    "hasGenre": {
        "keyword": "genre",
        "wd_pid": "P136",
        "wd_label": "genre",
        "alignment": "equivalent",
        "rationale": "P136 'genre' maps directly to prop:hasGenre. Both assert the genre of a musical work or artist.",
        "instance_pair": ("Q11975", "Q11399"),
    },
    "releasedAlbum": {
        "keyword": "discography",
        "wd_pid": "P358",
        "wd_label": "discography",
        "alignment": "subProperty",
        "rationale": "P358 'discography' groups all releases; prop:releasedAlbum is narrower (albums only).",
        "instance_pair": ("Q11975", "Q19953577"),
    },
    "signedTo": {
        "keyword": "record label",
        "wd_pid": "P264",
        "wd_label": "record label",
        "alignment": "equivalent",
        "rationale": "P264 'record label' is the canonical Wikidata property for artist-label affiliation.",
        "instance_pair": ("Q11975", "Q183412"),
    },
    "originatesFrom": {
        "keyword": "country of origin",
        "wd_pid": "P495",
        "wd_label": "country of origin",
        "alignment": "equivalent",
        "rationale": "P495 'country of origin' matches prop:originatesFrom for both artists and albums.",
        "instance_pair": ("Q11975", "Q30"),
    },
    "collaboratedWith": {
        "keyword": "performer",
        "wd_pid": "P175",
        "wd_label": "performer",
        "alignment": "related",
        "rationale": "P175 links a work to a performer, not artist-to-artist. No exact Wikidata equivalent exists for direct collaboration.",
        "instance_pair": None,
    },
    "influencedBy": {
        "keyword": "influenced by",
        "wd_pid": "P737",
        "wd_label": "influenced by",
        "alignment": "equivalent",
        "rationale": "P737 'influenced by' is a direct semantic match to prop:influencedBy.",
        "instance_pair": ("Q11975", "Q130070"),
    },
    "producedBy": {
        "keyword": "performer",
        "wd_pid": "P175",
        "wd_label": "performer",
        "alignment": "subProperty",
        "rationale": "P175 'performer' is the closest match for album->artist. prop:producedBy is narrower (emphasises production role).",
        "instance_pair": ("Q19953577", "Q11975"),
    },
    "birthYear": {
        "keyword": "date of birth",
        "wd_pid": "P569",
        "wd_label": "date of birth",
        "alignment": "subProperty",
        "rationale": "P569 is a full timestamp; prop:birthYear is year-only (coarser granularity).",
        "instance_pair": None,
    },
    "activeFrom": {
        "keyword": "work period start",
        "wd_pid": "P2031",
        "wd_label": "work period (start)",
        "alignment": "equivalent",
        "rationale": "P2031 'work period (start)' directly matches prop:activeFrom.",
        "instance_pair": None,
    },
    "releaseYear": {
        "keyword": "publication date",
        "wd_pid": "P577",
        "wd_label": "publication date",
        "alignment": "subProperty",
        "rationale": "P577 is a full timestamp; prop:releaseYear stores year only.",
        "instance_pair": None,
    },
    "albumRating": {
        "keyword": "review score",
        "wd_pid": "P444",
        "wd_label": "review score",
        "alignment": "related",
        "rationale": "P444 stores review scores as strings; prop:albumRating is a typed decimal. Related but different datatype semantics.",
        "instance_pair": None,
    },
    "foundedYear": {
        "keyword": "inception",
        "wd_pid": "P571",
        "wd_label": "inception",
        "alignment": "subProperty",
        "rationale": "P571 'inception' is a full timestamp; prop:foundedYear is year-only.",
        "instance_pair": None,
    },
    "partOfGenre": {
        "keyword": "subclass of",
        "wd_pid": "P279",
        "wd_label": "subclass of",
        "alignment": "related",
        "rationale": "P279 is the generic subclass relation; prop:partOfGenre is genre-specific.",
        "instance_pair": None,
    },
}
print(f"Alignment plan covers {len(ALIGNMENT_PLAN)} predicates.")

Alignment plan covers 14 predicates.


## 4. Run SPARQL label searches & instance validations

In [6]:
alignment_results = []

for prop_name, plan in ALIGNMENT_PLAN.items():
    prop_uri  = PROP[prop_name]
    wd_pid    = plan["wd_pid"]
    wd_label  = plan["wd_label"]
    keyword   = plan["keyword"]
    alignment = plan["alignment"]

    print(f"\n{'='*65}")
    print(f"  prop:{prop_name}")
    print(f"  Keyword search: '{keyword}'")

    # Strategy 1: Label search
    q1 = label_search_query(keyword, limit=8)
    rows1 = run_sparql(q1)
    time.sleep(1.0)

    if rows1:
        print(f"  Top label-search candidates:")
        for r in rows1[:5]:
            pid = r["prop"].split("/")[-1] if r["prop"] else "?"
            print(f"    {pid:<10} {r['propLabel']}")
    else:
        print("  [no results from label search]")

    # Strategy 2: Instance validation
    pair = plan.get("instance_pair")
    if pair:
        s_qid, o_qid = pair
        q2 = instance_validation_query(s_qid, o_qid, limit=10)
        rows2 = run_sparql(q2)
        time.sleep(1.0)
        if rows2:
            print(f"  Instance validation ({s_qid} -> {o_qid}):")
            for r in rows2:
                pid = r["prop"].split("/")[-1] if r["prop"] else "?"
                print(f"    {pid:<10} {r['propLabel']}")
        else:
            print("  [no instance properties returned]")

    print(f"  Decision: prop:{prop_name}  --[{alignment}]-->  wdt:{wd_pid} ('{wd_label}')")
    print(f"  Rationale: {plan['rationale']}")

    alignment_results.append({
        "Private Predicate": f"prop:{prop_name}",
        "Keyword Used":      keyword,
        "Wikidata PID":      f"wdt:{wd_pid}",
        "Wikidata Label":    wd_label,
        "Alignment Type":   alignment,
        "Rationale":        plan["rationale"],
    })

print(f"\nCompleted {len(alignment_results)} predicate alignments.")


  prop:wonAward
  Keyword search: 'award'
  Top label-search candidates:
    P12262     Abandonware-France award ID
    P6145      Academy Awards Database film ID
    P6150      Academy Awards Database nominee ID
    P5645      Académie française award winner ID
    P7184      Awards & Winners artist ID
  [no instance properties returned]
  Decision: prop:wonAward  --[equivalent]-->  wdt:P166 ('award received')
  Rationale: P166 'award received' is the standard Wikidata property for winning an award. Semantically identical.

  prop:hasGenre
  Keyword search: 'genre'
  Top label-search candidates:
    P11564     AllMovie genre ID
    P9185      AllMusic genre/style ID
    P13703     Deaf Movie Database genre ID
    P9218      Discogs genre ID
    P12947     GameFAQs genre ID
  [no instance properties returned]
  Decision: prop:hasGenre  --[equivalent]-->  wdt:P136 ('genre')
  Rationale: P136 'genre' maps directly to prop:hasGenre. Both assert the genre of a musical work or artist.

  p

## 5. Write alignment axioms into the graph

In [7]:
ALIGNMENT_MAP = {
    "equivalent":   OWL.equivalentProperty,
    "subProperty":  RDFS.subPropertyOf,
    "related":      SKOS.relatedMatch,
}

axioms_added = 0
for prop_name, plan in ALIGNMENT_PLAN.items():
    prop_uri = PROP[prop_name]
    wdt_uri  = WDT[plan["wd_pid"]]
    align    = plan["alignment"]

    if align == "superProperty":
        g.add((wdt_uri, RDFS.subPropertyOf, prop_uri))
    else:
        axiom_prop = ALIGNMENT_MAP.get(align)
        if axiom_prop:
            g.add((prop_uri, axiom_prop, wdt_uri))

    g.add((prop_uri, RDFS.comment,
           Literal(f"Aligned to wdt:{plan['wd_pid']} ({plan['wd_label']}) "
                   f"via {align}. {plan['rationale']}")))
    axioms_added += 1
    print(f"  prop:{prop_name:<25} --[{align}]--> wdt:{plan['wd_pid']}")

print(f"\n{axioms_added} alignment axioms written to graph.")

  prop:wonAward                  --[equivalent]--> wdt:P166
  prop:hasGenre                  --[equivalent]--> wdt:P136
  prop:releasedAlbum             --[subProperty]--> wdt:P358
  prop:signedTo                  --[equivalent]--> wdt:P264
  prop:originatesFrom            --[equivalent]--> wdt:P495
  prop:collaboratedWith          --[related]--> wdt:P175
  prop:influencedBy              --[equivalent]--> wdt:P737
  prop:producedBy                --[subProperty]--> wdt:P175
  prop:birthYear                 --[subProperty]--> wdt:P569
  prop:activeFrom                --[equivalent]--> wdt:P2031
  prop:releaseYear               --[subProperty]--> wdt:P577
  prop:albumRating               --[related]--> wdt:P444
  prop:foundedYear               --[subProperty]--> wdt:P571
  prop:partOfGenre               --[related]--> wdt:P279

14 alignment axioms written to graph.


## 6. Alignment summary table

In [8]:
df = pd.DataFrame(alignment_results)
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.max_rows", 30)
display(df[["Private Predicate","Wikidata PID","Wikidata Label","Alignment Type","Rationale"]])

print("\nAlignment type distribution:")
print(df["Alignment Type"].value_counts().to_string())

df.to_csv("predicate_alignment_table.csv", index=False)
print("\nSaved -> predicate_alignment_table.csv")

,Private Predicate,Wikidata PID,Wikidata Label,Alignment Type,Rationale
0,prop:wonAward,wdt:P166,award received,equivalent,P166 'award received' is the standard Wikidata property for winnin...
1,prop:hasGenre,wdt:P136,genre,equivalent,P136 'genre' maps directly to prop:hasGenre. Both assert the genre...
2,prop:releasedAlbum,wdt:P358,discography,subProperty,P358 'discography' groups all releases; prop:releasedAlbum is narr...
3,prop:signedTo,wdt:P264,record label,equivalent,P264 'record label' is the canonical Wikidata property for artist-...
4,prop:originatesFrom,wdt:P495,country of origin,equivalent,P495 'country of origin' matches prop:originatesFrom for both arti...
5,prop:collaboratedWith,wdt:P175,performer,related,"P175 links a work to a performer, not artist-to-artist. No exact W..."
6,prop:influencedBy,wdt:P737,influenced by,equivalent,P737 'influenced by' is a direct semantic match to prop:influencedBy.
7,prop:producedBy,wdt:P175,performer,subProperty,P175 'performer' is the closest match for album->artist. prop:prod...
8,prop:birthYear,wdt:P569,date of birth,subProperty,P569 is a full timestamp; prop:birthYear is year-only (coarser gra...
9,prop:activeFrom,wdt:P2031,work period (start),equivalent,P2031 'work period (start)' directly matches prop:activeFrom.



Alignment type distribution:
Alignment Type
equivalent     6
subProperty    5
related        3

Saved -> predicate_alignment_table.csv


## 7. SPARQL verification — query the aligned KB

In [9]:
q = """
PREFIX owl:  <http://www.w3.org/2002/07/owl#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>

SELECT ?privateProp ?alignmentType ?externalProp WHERE {
  {
    ?privateProp owl:equivalentProperty ?externalProp .
    BIND("equivalentProperty" AS ?alignmentType)
  } UNION {
    ?privateProp rdfs:subPropertyOf ?externalProp .
    BIND("subPropertyOf" AS ?alignmentType)
  } UNION {
    ?privateProp skos:relatedMatch ?externalProp .
    BIND("relatedMatch" AS ?alignmentType)
  }
  FILTER(STRSTARTS(STR(?privateProp), "http://example.org/music/prop/"))
}
ORDER BY ?alignmentType ?privateProp
"""
print(f"{'Private Predicate':<30} {'Alignment':<22} {'External Property'}")
print("-" * 90)
for row in g.query(q):
    priv = str(row.privateProp).split("/")[-1]
    ext  = str(row.externalProp).split("/")[-1]
    print(f"  prop:{priv:<26} {str(row.alignmentType):<22} wdt:{ext}")

Private Predicate              Alignment              External Property
------------------------------------------------------------------------------------------
  prop:activeFrom                 equivalentProperty     wdt:P2031
  prop:hasGenre                   equivalentProperty     wdt:P136
  prop:influencedBy               equivalentProperty     wdt:P737
  prop:originatesFrom             equivalentProperty     wdt:P495
  prop:signedTo                   equivalentProperty     wdt:P264
  prop:wonAward                   equivalentProperty     wdt:P166
  prop:albumRating                relatedMatch           wdt:P444
  prop:collaboratedWith           relatedMatch           wdt:P175
  prop:partOfGenre                relatedMatch           wdt:P279
  prop:birthYear                  subPropertyOf          wdt:P569
  prop:foundedYear                subPropertyOf          wdt:P571
  prop:producedBy                 subPropertyOf          wdt:P175
  prop:releaseYear                subPropert

## 8. Serialize final KB

In [10]:
g.serialize(destination="music_kb_aligned.ttl", format="turtle")
print(f"Saved -> music_kb_aligned.ttl  ({len(g)} triples total)")

with open("music_kb_aligned.ttl") as f:
    lines = f.readlines()

align_lines = [l for l in lines
               if any(x in l for x in ["equivalentProperty","subPropertyOf","relatedMatch"])
               and "prop/" in l]
print(f"\nAlignment triples in TTL ({len(align_lines)}):")
for l in align_lines:
    print(" ", l.rstrip())

Saved -> music_kb_aligned.ttl  (876 triples total)

Alignment triples in TTL (0):


In [1]:
!jupyter nbconvert --to html step3_predicate_alignment.ipynb


[NbConvertApp] Converting notebook step3_predicate_alignment.ipynb to html
[NbConvertApp] Writing 350739 bytes to step3_predicate_alignment.html
